In [20]:
import os
from pathlib import Path
import duckdb

# Define input and output paths
INPUT_DIR = "/home/jovyan/work/portfolio/data/nhl/data/landing/schedule/season=20252026"      # Directory containing your .json.gz files
OUTPUT_DIR = "/home/jovyan/work/portfolio/data/nhl/data/warehouse/schedule/season=20252026"  # Target directory for dbt source files
print(os.getcwd())
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Initialize DuckDB connection
con = duckdb.connect()

print("Reading and converting json.gz files to Parquet...")

# DuckDB can read gzipped line-delimited JSON directly using wildcards
query = f"""
    COPY (
        SELECT 
            *,
            filename AS _source_file,
            CURRENT_TIMESTAMP AS _loaded_at
        FROM read_json_auto(
            '{INPUT_DIR}/*.json.gz', 
            filename=true, 
            union_by_name=true
        )
    ) TO '{OUTPUT_DIR}' (
        FORMAT PARQUET, 
        COMPRESSION 'SNAPPY',
        PER_THREAD_OUTPUT true
    );
"""

con.execute(query)
print(f"Success! Parquet files written to: {OUTPUT_DIR}")

/home/jovyan
Reading and converting json.gz files to Parquet...
Success! Parquet files written to: /home/jovyan/work/portfolio/data/nhl/data/warehouse/schedule/season=20252026


In [ ]:
import os
import shutil
import subprocess

# 1. Print Current Working Directory
cwd = os.getcwd()
print("=" * 60)
print(f"CURRENT WORKING DIRECTORY:\n  {cwd}")
print(f"  Writable? -> {os.access(cwd, os.W_OK)}")
print("=" * 60)

# 2. Inspect Folders and Files in Current Directory
print("\nCONTENTS OF CURRENT DIRECTORY:")
try:
    for item in os.listdir(cwd):
        full_path = os.path.join(cwd, item)
        is_dir = os.path.isdir(full_path)
        is_writable = os.access(full_path, os.W_OK)
        item_type = "DIR " if is_dir else "FILE"
        print(f"  [{item_type}] {item:<30} (Writable: {is_writable})")
except Exception as e:
    print(f"  Error reading directory: {e}")

# 3. Check /home/jovian/ Path Permissions Specifically
print("\n" + "=" * 60)
print("PATH WRITABILITY INSPECTOR:")
target_paths = [
    "/home/jovian",
    "/home/jovian/work",
    "/home/jovian/work/data",
    "/home/jovian/work/data/landing",
    "/home/jovian/work/data/landing/boxscore/season=20252026"
]

for path in target_paths:
    exists = os.path.exists(path)
    writable = os.access(path, os.W_OK) if exists else "N/A (Path does not exist)"
    print(f"  {path:<55} -> Exists: {str(exists):<5} | Writable: {writable}")

# 4. Check Volume Mount Points (df -h)
print("\n" + "=" * 60)
print("MOUNTED VOLUMES & DISK SPACE (df -h):")
try:
    df_output = subprocess.check_output(["df", "-h"], text=True)
    print(df_output)
except Exception as e:
    print(f"  Unable to run df command: {e}")

CURRENT WORKING DIRECTORY:
  /home/jovyan
  Writable? -> True

CONTENTS OF CURRENT DIRECTORY:
  [FILE] .bash_logout                   (Writable: True)
  [FILE] .bashrc                        (Writable: True)
  [FILE] .profile                       (Writable: True)
  [DIR ] .local                         (Writable: True)
  [DIR ] .ipython                       (Writable: True)
  [DIR ] .jupyter                       (Writable: True)
  [DIR ] data                           (Writable: True)
  [DIR ] warehouse                      (Writable: True)
  [DIR ] .cache                         (Writable: True)
  [DIR ] .config                        (Writable: True)
  [DIR ] .mamba                         (Writable: True)
  [DIR ] work                           (Writable: True)

PATH WRITABILITY INSPECTOR:
  /home/jovian                                            -> Exists: False | Writable: N/A (Path does not exist)
  /home/jovian/work                                       -> Exists: False | Wri

In [18]:
import os

# Check both possible paths with the corrected 'jovyan' spelling
path_in_home = "/home/jovyan/data/landing/boxscore/season=20252026"
path_in_work = "/home/jovyan/work/data/landing/boxscore/season=20252026"

print(f"Checking {path_in_home} -> Exists: {os.path.exists(path_in_home)}")
print(f"Checking {path_in_work} -> Exists: {os.path.exists(path_in_work)}")

Checking /home/jovyan/data/landing/boxscore/season=20252026 -> Exists: False
Checking /home/jovyan/work/data/landing/boxscore/season=20252026 -> Exists: False


In [19]:
import os

def find_season_dir(search_path):
    print(f"Searching inside: {search_path}")
    for root, dirs, files in os.walk(search_path):
        if "season=20252026" in dirs:
            full_path = os.path.join(root, "season=20252026")
            print(f"  FOUND: {full_path}")
            # Check for json.gz files in found directory
            gz_files = [f for f in os.listdir(full_path) if f.endswith('.json.gz')]
            print(f"    Contains {len(gz_files)} .json.gz file(s)")

# Search inside both work (the mount) and current home directory
find_season_dir("/home/jovyan/work")
find_season_dir("/home/jovyan/data")

Searching inside: /home/jovyan/work
  FOUND: /home/jovyan/work/portfolio/data/nhl/data/landing/schedule/season=20252026
    Contains 32 .json.gz file(s)
  FOUND: /home/jovyan/work/portfolio/data/nhl/data/landing/boxscore/season=20252026
    Contains 15 .json.gz file(s)
  FOUND: /home/jovyan/work/portfolio/data/nhl/data/landing/play_by_play/season=20252026
    Contains 15 .json.gz file(s)
  FOUND: /home/jovyan/work/portfolio/data/nhl/data/landing/shift_chart/season=20252026
    Contains 15 .json.gz file(s)
  FOUND: /home/jovyan/work/data/warehouse/boxscore/season=20252026
    Contains 0 .json.gz file(s)
Searching inside: /home/jovyan/data
  FOUND: /home/jovyan/data/warehouse/boxscore/season=20252026
    Contains 0 .json.gz file(s)
